# 02 - Silver validation report

Independent checks against the `slv_*` tables. This notebook writes nothing - it re-derives the
answers a second way and compares.

The point is that a Silver notebook which runs without error has proved nothing. It has to be
checked against something it did not compute itself.

Run `00_Bronze_Ingest_Excel.ipynb` and `01_Silver_Cleanse_Standardise.ipynb` first.

In [ ]:
BRONZE_PREFIX = "brz_"
SILVER_PREFIX = "slv_"

from pyspark.sql import functions as F

failures = []


def check(name, passed, detail=""):
    """Record a pass/fail so the last cell can summarise."""
    status = "PASS" if passed else "FAIL"
    if not passed:
        failures.append(name)
    print(f"[{status}] {name}" + (f"   {detail}" if detail else ""))
    return passed

print("Ready.")

## 1. No rows were lost or invented

Silver flags rather than deletes, so with `DEDUPE_MODE = "flag"` every Bronze row must still be
there. If this fails, a join fanned out or a filter slipped in.

In [ ]:
bronze_sales = (spark.table(BRONZE_PREFIX + "pharma_sales_2025").count()
                + spark.table(BRONZE_PREFIX + "pharma_sales_2026").count())
silver_sales = spark.table(SILVER_PREFIX + "pharma_sales").count()

check("sales row count preserved", bronze_sales == silver_sales,
      f"bronze {bronze_sales:,} vs silver {silver_sales:,}")

for bronze_name, silver_name in [("pharma_products", "pharma_products"),
                                 ("pharma_markets", "pharma_markets"),
                                 ("pharma_customers", "pharma_customers"),
                                 ("pharma_forecast", "pharma_forecast"),
                                 ("pharma_budget", "pharma_budget"),
                                 ("pharma_inventory", "pharma_inventory"),
                                 ("pharma_opex", "pharma_opex")]:
    bronze_count = spark.table(BRONZE_PREFIX + bronze_name).count()
    silver_count = spark.table(SILVER_PREFIX + silver_name).count()
    check(f"{silver_name} row count preserved", bronze_count == silver_count,
          f"{bronze_count:,} vs {silver_count:,}")

## 2. Every string was actually cleaned

Three things that should now be impossible anywhere in Silver: untrimmed values, empty strings
masquerading as values, and more than one casing of the same currency or scenario.

In [ ]:
sales = spark.table(SILVER_PREFIX + "pharma_sales")
customers = spark.table(SILVER_PREFIX + "pharma_customers")
markets = spark.table(SILVER_PREFIX + "pharma_markets")

untrimmed = sales.filter(
    (F.col("Currency") != F.trim(F.col("Currency")))
    | (F.col("ScenarioVersion") != F.trim(F.col("ScenarioVersion")))).count()
check("no untrimmed strings in sales", untrimmed == 0, f"{untrimmed} found")

empty_strings = customers.filter(
    (F.col("CustomerType") == "") | (F.col("Country") == "")).count()
check("no empty-string values in customers", empty_strings == 0, f"{empty_strings} found")

currencies = sorted(r[0] for r in sales.select("Currency").distinct().collect() if r[0])
check("currencies standardised to upper case", all(c == c.upper() for c in currencies),
      str(currencies))

scenarios = sorted(r[0] for r in sales.select("ScenarioVersion").distinct().collect() if r[0])
check("scenario values canonical", set(scenarios) <= {"Actual", "Budget", "Forecast"},
      str(scenarios))

types = sorted(r[0] for r in customers.select("CustomerType").distinct().collect() if r[0])
check("customer types collapsed to one casing", len(types) == len(set(t.title() for t in types)),
      str(types))

countries = sorted(r[0] for r in markets.select("Country").distinct().collect() if r[0])
print("\nCountries:", countries)

## 3. Every date parsed

The source stored dates in five text formats. Any row where `TransactionDate` is null but the
Bronze value was not null means the parser is missing a pattern - which is a silent data loss, not
an error you would otherwise see.

In [ ]:
unparsed = sales.filter(F.col("TransactionDate").isNull()).count()
check("all transaction dates parsed", unparsed == 0, f"{unparsed} unparsed")

for table_name, column in [("pharma_forecast", "Month"), ("pharma_budget", "Month"),
                           ("pharma_inventory", "InventoryDate"), ("pharma_opex", "Month"),
                           ("pharma_products", "LaunchDate"), ("pharma_fx_rates", "RateMonth")]:
    nulls = spark.table(SILVER_PREFIX + table_name).filter(F.col(column).isNull()).count()
    check(f"{table_name}.{column} fully parsed", nulls == 0, f"{nulls} null")

print("\nDate range landed in sales:")
sales.select(F.min("TransactionDate").alias("From"),
             F.max("TransactionDate").alias("To")).show()

print("Rows per year - 2026 should only cover Jan-Mar:")
sales.groupBy("TransactionYear", "TransactionMonth").count() \
     .orderBy("TransactionYear", "TransactionMonth").show(20)

## 4. Recalculate the revenue arithmetic independently

Silver flagged the rows it thought were wrong. This recomputes the same arithmetic from scratch
and checks the two agree - if Silver's tolerance or column references were wrong, this disagrees.

In [ ]:
recalc = (sales
    .filter("UnitsSold IS NOT NULL AND UnitPriceLocal IS NOT NULL")
    .withColumn("_gross", F.round(F.col("UnitsSold") * F.col("UnitPriceLocal"), 2))
    .withColumn("_net", F.round(F.col("GrossSalesLocal") - F.col("DiscountLocal"), 2))
    .withColumn("_grossBroken", F.abs(F.col("_gross") - F.col("GrossSalesLocal")) > 0.01)
    .withColumn("_netBroken", F.abs(F.col("_net") - F.col("NetSalesLocal")) > 0.01))

independent_broken = recalc.filter("_grossBroken OR _netBroken").count()
silver_flagged = sales.filter(
    F.array_contains("DataQualityFlags", "GROSS_SALES_MISMATCH")
    | F.array_contains("DataQualityFlags", "NET_SALES_MISMATCH")).count()

check("revenue arithmetic agrees with Silver's own flags",
      independent_broken == silver_flagged,
      f"independent {independent_broken} vs flagged {silver_flagged}")

print("\nReturns keep the arithmetic intact (negative units, negative money):")
(sales.filter("IsReturn")
      .select("TransactionID", "UnitsSold", "UnitPriceLocal", "GrossSalesLocal",
              "DiscountLocal", "NetSalesLocal")
      .show(5, truncate=False))

## 5. Currency conversion went the right way

`RateToEUR` is local currency per 1 EUR, so dividing is correct. The tell is direction: GBP is
worth more than EUR, so a GBP amount must get **larger** in EUR; AED is worth less, so an AED
amount must get **smaller**. Multiplying instead of dividing inverts both, and nothing else in the
pipeline would catch it.

In [ ]:
conversion = (sales.filter("NetSalesLocal > 0 AND NetSalesEUR IS NOT NULL")
    .groupBy("Currency")
    .agg(F.round(F.sum("NetSalesLocal") / 1e6, 2).alias("NetLocalM"),
         F.round(F.sum("NetSalesEUR") / 1e6, 2).alias("NetEURm"),
         F.round(F.avg("RateToEUR"), 4).alias("AvgRate"))
    .orderBy("Currency"))
conversion.show()

rows = {r["Currency"]: r for r in conversion.collect()}

if "GBP" in rows:
    check("GBP converts upward into EUR", rows["GBP"]["NetEURm"] > rows["GBP"]["NetLocalM"],
          f"{rows['GBP']['NetLocalM']}m GBP -> {rows['GBP']['NetEURm']}m EUR")
if "AED" in rows:
    check("AED converts downward into EUR", rows["AED"]["NetEURm"] < rows["AED"]["NetLocalM"],
          f"{rows['AED']['NetLocalM']}m AED -> {rows['AED']['NetEURm']}m EUR")
if "EUR" in rows:
    check("EUR is unchanged", abs(rows["EUR"]["NetEURm"] - rows["EUR"]["NetLocalM"]) < 0.01)

missing_fx = sales.filter("RateToEUR IS NULL AND Currency IS NOT NULL").count()
check("every row found an FX rate", missing_fx == 0, f"{missing_fx} without a rate")

## 6. Referential integrity

Every non-null key in a fact table must exist in its dimension. Rows with a *null* key are a
separate problem, already flagged - this checks for keys that look valid but match nothing.

In [ ]:
products = spark.table(SILVER_PREFIX + "pharma_products")
customers = spark.table(SILVER_PREFIX + "pharma_customers")

orphans = {
    "sales -> products": sales.filter("ProductID IS NOT NULL").join(
        products, "ProductID", "left_anti").count(),
    "sales -> markets": sales.filter("MarketID IS NOT NULL").join(
        markets, "MarketID", "left_anti").count(),
    "sales -> customers": sales.filter("CustomerID IS NOT NULL").join(
        customers, "CustomerID", "left_anti").count(),
    "forecast -> products": spark.table(SILVER_PREFIX + "pharma_forecast").filter(
        "ProductID IS NOT NULL").join(products, "ProductID", "left_anti").count(),
    "budget -> markets": spark.table(SILVER_PREFIX + "pharma_budget").filter(
        "MarketID IS NOT NULL").join(markets, "MarketID", "left_anti").count(),
    "inventory -> products": spark.table(SILVER_PREFIX + "pharma_inventory").filter(
        "ProductID IS NOT NULL").join(products, "ProductID", "left_anti").count(),
    "opex -> markets": spark.table(SILVER_PREFIX + "pharma_opex").filter(
        "MarketID IS NOT NULL").join(markets, "MarketID", "left_anti").count(),
}

for relationship, count in orphans.items():
    check(f"no orphans: {relationship}", count == 0, f"{count} orphan rows")

print("\nRows carrying a null key (flagged, not orphaned):")
(sales.select(
    F.sum(F.col("ProductID").isNull().cast("int")).alias("NullProductID"),
    F.sum(F.col("MarketID").isNull().cast("int")).alias("NullMarketID"),
    F.sum(F.col("CustomerID").isNull().cast("int")).alias("NullCustomerID")).show())

## 7. Schema drift was reconciled, not papered over

The 2026 file renamed the price column, moved `CustomerID`, upper-cased the currency header and
added two columns. Proof that it worked:

- both years carry a populated `UnitPriceLocal` (the rename resolved)
- 2026 has `SalesChannel` and `SourceSystem`, 2025 has them as null (the new columns aligned)
- both years carry a populated `CustomerID` (position is irrelevant)

In [ ]:
drift = (sales.groupBy("TransactionYear").agg(
    F.count(F.lit(1)).alias("Rows"),
    F.sum(F.col("UnitPriceLocal").isNotNull().cast("int")).alias("HasUnitPrice"),
    F.sum(F.col("CustomerID").isNotNull().cast("int")).alias("HasCustomerID"),
    F.sum(F.col("Currency").isNotNull().cast("int")).alias("HasCurrency"),
    F.sum(F.col("SalesChannel").isNotNull().cast("int")).alias("HasSalesChannel"),
    F.sum(F.col("SourceSystem").isNotNull().cast("int")).alias("HasSourceSystem"))
    .orderBy("TransactionYear"))
drift.show()

by_year = {r["TransactionYear"]: r for r in drift.collect()}

check("2025 price column resolved despite the 2026 rename",
      by_year[2025]["HasUnitPrice"] > 0 and by_year[2026]["HasUnitPrice"] > 0)
check("CustomerID survived moving column position",
      by_year[2026]["HasCustomerID"] > 0)
check("CURRENCY header matched Currency",
      by_year[2026]["HasCurrency"] > 0)
check("SalesChannel only populated for 2026",
      by_year[2025]["HasSalesChannel"] == 0 and by_year[2026]["HasSalesChannel"] > 0)

print("\nNew dimensions that only exist from 2026 onward:")
(sales.filter("SalesChannel IS NOT NULL")
      .groupBy("SourceSystem", "SalesChannel").count()
      .orderBy("SourceSystem", "SalesChannel").show())

## 8. Cross-foot inventory against sales

The inventory file reports its own `UnitsSold` per product, market and month. Aggregating the
sales table should land close to it.

**It will not match exactly, and neither direction of the gap is an error.** The cell below builds
a bridge instead of asserting a single number, because each exclusion moves the total a different
way:

- **duplicate rows inflate it** - the same shipment counted twice
- **volume outliers inflate it** - a few tender orders large enough to swamp a monthly total
- **returns deflate it** - negative units against a sale booked earlier

Removing all three *overshoots*, and it has to. A return reverses part of a sale the warehouse
really did ship, and an outlier is a real order that happened to be large - so excluding those rows
throws away legitimate volume along with the distortion.

That gives the useful test: the inventory figure should sit **between** the raw sales total and the
fully-excluded one. If it falls outside that bracket, something is genuinely wrong. Chasing the
residual to zero would mean tuning the filters until they flatter the answer.

In [ ]:
inventory = spark.table(SILVER_PREFIX + "pharma_inventory")

sales_by_month = (sales
    .filter("DataQualityStatus <> 'Quarantine' AND TransactionYear = 2025")
    .groupBy("ProductID", "MarketID", "MonthKey")
    .agg(F.sum("UnitsSold").alias("SalesUnits")))

inventory_by_month = (inventory
    .groupBy("ProductID", "MarketID", "MonthKey")
    .agg(F.sum("UnitsSold").alias("InventoryUnits")))

comparison = (inventory_by_month
    .join(sales_by_month, ["ProductID", "MarketID", "MonthKey"], "left")
    .withColumn("SalesUnits", F.coalesce(F.col("SalesUnits"), F.lit(0)))
    .withColumn("Variance", F.col("SalesUnits") - F.col("InventoryUnits")))

def units_2025(extra_filter=None):
    """Total 2025 units under a given exclusion, null keys included - they are real units."""
    scoped = sales.filter("TransactionYear = 2025")
    if extra_filter:
        scoped = scoped.filter(extra_filter)
    total = scoped.agg(F.sum("UnitsSold")).collect()[0][0]
    return int(total or 0)


inventory_units = int(inventory.agg(F.sum("UnitsSold")).collect()[0][0])

step_all = units_2025()
step_no_duplicates = units_2025("NOT IsDuplicateRow")
step_no_outliers = units_2025("NOT IsDuplicateRow AND NOT IsVolumeOutlier")
step_no_returns = units_2025("NOT IsDuplicateRow AND NOT IsVolumeOutlier AND NOT IsReturn")

print(f"{'Sales table, every 2025 row':<42}{step_all:>12,}")
print(f"{'  less duplicate rows':<42}{step_no_duplicates:>12,}"
      f"{step_no_duplicates - step_all:>12,}")
print(f"{'  less volume outliers':<42}{step_no_outliers:>12,}"
      f"{step_no_outliers - step_no_duplicates:>12,}")
print(f"{'  less returns':<42}{step_no_returns:>12,}"
      f"{step_no_returns - step_no_outliers:>12,}")
print(f"{'-' * 66}")
print(f"{'Inventory file says sold':<42}{inventory_units:>12,}")

residual = step_no_returns - inventory_units
print(f"{'Residual vs fully-excluded sales':<42}{residual:>12,}"
      f"   ({residual / inventory_units:+.2%})")

check("inventory sits between the raw and the fully-excluded sales total",
      step_no_returns <= inventory_units <= step_all,
      f"{step_no_returns:,} <= {inventory_units:,} <= {step_all:,}")
check("residual after all three exclusions is under 2%",
      abs(residual) / inventory_units < 0.02,
      f"{residual / inventory_units:+.2%}")

comparison = comparison.withColumn(
    "VariancePct",
    F.when(F.col("InventoryUnits") > 0,
           F.round(F.col("Variance") * 100 / F.col("InventoryUnits"), 1)))

print("\nLargest per-combination gaps - each one is worth a look:")
outlier_months = (sales
    .filter("IsVolumeOutlier AND TransactionYear = 2025")
    .select("ProductID", "MarketID", "MonthKey").distinct()
    .withColumn("HasFlaggedOutlier", F.lit(True)))

top_gaps = (comparison.orderBy(F.abs(F.col("Variance")).desc()).limit(10)
    .join(outlier_months, ["ProductID", "MarketID", "MonthKey"], "left")
    .withColumn("HasFlaggedOutlier", F.coalesce(F.col("HasFlaggedOutlier"), F.lit(False)))
    .orderBy(F.abs(F.col("Variance")).desc()))

top_gaps.select("ProductID", "MarketID", "MonthKey", "InventoryUnits",
                "SalesUnits", "Variance", "VariancePct", "HasFlaggedOutlier").show(10)

explained = top_gaps.filter("HasFlaggedOutlier").count()
print(f"{explained} of the 10 largest gaps sit in a month containing a flagged outlier.")
check("the largest gaps are explained by flagged outliers", explained >= 8,
      f"{explained}/10 explained")

## 9. Plan versus actual is comparable

Forecast and budget are the same grain, so they should cover the same product-market pairs. A gap
here means the Gold `FactPlan` union will have holes in one scenario and not the other.

In [ ]:
forecast = spark.table(SILVER_PREFIX + "pharma_forecast")
budget = spark.table(SILVER_PREFIX + "pharma_budget")

forecast_keys = forecast.select("ProductID", "MarketID").distinct()
budget_keys = budget.select("ProductID", "MarketID").distinct()

only_forecast = forecast_keys.join(budget_keys, ["ProductID", "MarketID"], "left_anti").count()
only_budget = budget_keys.join(forecast_keys, ["ProductID", "MarketID"], "left_anti").count()

check("forecast and budget cover the same product-market pairs",
      only_forecast == 0 and only_budget == 0,
      f"forecast-only {only_forecast}, budget-only {only_budget}")

# float() matters: Spark returns decimal sums as Python Decimal, which will not
# divide against a float literal
plan = float(forecast.filter("DataQualityStatus <> 'Quarantine'")
                     .agg(F.sum("RevenueForecastEUR")).collect()[0][0])
budget_total = float(budget.filter("DataQualityStatus <> 'Quarantine'")
                           .agg(F.sum("BudgetRevenueEUR")).collect()[0][0])
actual = float(sales.filter("DataQualityStatus <> 'Quarantine' AND TransactionYear = 2025")
                    .agg(F.sum("NetSalesEUR")).collect()[0][0])

print(f"\nFY2025 actual net sales   : EUR {actual / 1e6:8.1f}m")
print(f"FY2026 forecast revenue   : EUR {plan / 1e6:8.1f}m  ({plan / actual - 1:+.1%} vs actual)")
print(f"FY2026 budget revenue     : EUR {budget_total / 1e6:8.1f}m  "
      f"({budget_total / plan - 1:+.1%} vs forecast)")

check("FY2026 forecast is within a sane range of FY2025", 0.9 < plan / actual < 1.4)

## 10. Data-quality summary and Gold readiness

What Gold should actually load: everything except `Quarantine`. Note how small that exclusion is -
if this ever climbs above a few percent, the source has changed and the pipeline needs attention
rather than a bigger filter.

In [ ]:
print("Flag counts across the Silver layer:\n")
(spark.table(SILVER_PREFIX + "data_quality_summary")
      .filter("DataQualityFlag <> '(clean row)'")
      .orderBy("SilverTable", F.desc("RowCount"))
      .show(40, truncate=False))

print("Gold-eligible rows per table:\n")
for table_name in ["pharma_sales", "pharma_forecast", "pharma_budget",
                   "pharma_inventory", "pharma_opex", "pharma_products",
                   "pharma_customers", "pharma_markets"]:
    table = spark.table(SILVER_PREFIX + table_name)
    total = table.count()
    eligible = table.filter("DataQualityStatus <> 'Quarantine'").count()
    print(f"  {table_name:22s} {eligible:>7,} of {total:>7,} "
          f"({eligible / total:6.1%}) load to Gold")

## 11. Result

In [ ]:
print("=" * 60)
if failures:
    print(f"{len(failures)} CHECK(S) FAILED:")
    for name in failures:
        print("  -", name)
else:
    print("ALL CHECKS PASSED - the Silver layer is ready for Gold.")
print("=" * 60)

## Appendix - the same checks in SQL

Fabric notebooks accept `%%sql`, so these tables are queryable directly. Handy for a quick look
without writing PySpark.

In [ ]:
%%sql
-- top ten products by FY2025 net sales, excluding quarantined rows
SELECT  p.ProductName,
        p.TherapeuticArea,
        ROUND(SUM(s.NetSalesEUR) / 1000000, 2) AS NetSalesEURm,
        ROUND(SUM(s.CostEUR)     / 1000000, 2) AS CostEURm
FROM    slv_pharma_sales      AS s
JOIN    slv_pharma_products   AS p  ON p.ProductID = s.ProductID
WHERE   s.DataQualityStatus <> 'Quarantine'
  AND   s.TransactionYear = 2025
GROUP BY p.ProductName, p.TherapeuticArea
ORDER BY NetSalesEURm DESC

In [ ]:
%%sql
-- every row the pipeline wants a human to look at
SELECT  DataQualityStatus,
        DataQualityFlags,
        COUNT(*) AS Rows
FROM    slv_pharma_sales
WHERE   DataQualityFlagCount > 0
GROUP BY DataQualityStatus, DataQualityFlags
ORDER BY Rows DESC